In [ ]:
#| default_exp renderers.comfyui

# renderers.comfyui

> Renderer for a local ComfyUI server via its REST API.
>
> Supports LoRA, negative prompts, and reference image input (via IP-Adapter or
> similar nodes in the workflow). Requires a running ComfyUI instance and a
> workflow JSON exported in API format.
>
> Config: `renderer.comfyui.base_url` and `renderer.comfyui.workflow_template_path`.

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations
import json
import time
import uuid
from pathlib import Path

import httpx

from manhualizer.config import OutputConfig, RendererConfig
from manhualizer.models import Panel, RenderResult
from manhualizer.render import BaseRenderer, ModelSpec

In [ ]:
#| export
# Node keys used to patch the workflow template.
# These must match the node titles in your ComfyUI workflow.
# Override by subclassing or setting these class attributes.
_PROMPT_NODE_TITLE = "positive_prompt"
_NEGATIVE_NODE_TITLE = "negative_prompt"
_WIDTH_NODE_TITLE = "empty_latent_image"
_LORA_NODE_TITLE = "lora_loader"
_REF_IMAGE_NODE_TITLE = "reference_image"

In [ ]:
#| export
class ComfyUIRenderer(BaseRenderer):
    """Image generation via a local ComfyUI server.

    Workflow is loaded from a JSON file in ComfyUI API format. The renderer
    patches specific nodes by title (positive prompt, negative prompt,
    dimensions, LoRA, reference image) before submitting each job.

    Polling waits for job completion; timeout is configurable.

    Config:
        renderer.comfyui.base_url           — ComfyUI server URL
        renderer.comfyui.workflow_template_path — path to workflow JSON
        renderer.comfyui.poll_interval      — seconds between status checks
        renderer.comfyui.timeout            — max wait seconds per panel
    """

    def __init__(self, model_spec: ModelSpec, config: RendererConfig):
        super().__init__(model_spec, config)
        self._model_cfg = config.comfyui
        self._workflow_template: dict | None = None

    def _load_workflow(self) -> dict:
        if self._workflow_template is None:
            path = Path(self._model_cfg.workflow_template_path)
            if not path.exists():
                raise FileNotFoundError(
                    f"ComfyUI workflow not found: {path}. "
                    f"Set renderer.comfyui.workflow_template_path in your config."
                )
            self._workflow_template = json.loads(path.read_text())
        return json.loads(json.dumps(self._workflow_template))  # deep copy

    def _patch_workflow(
        self,
        workflow: dict,
        panel: Panel,
        output_cfg: OutputConfig,
        negative_prompt: str = "",
        reference_image_b64: str | None = None,
    ) -> dict:
        """Patch workflow nodes with panel-specific values."""
        w, h = output_cfg.resolved_dimensions()

        for node in workflow.values():
            title = node.get("_meta", {}).get("title", "")
            inputs = node.get("inputs", {})

            if title == _PROMPT_NODE_TITLE and "text" in inputs:
                inputs["text"] = panel.visual_prompt
            elif title == _NEGATIVE_NODE_TITLE and "text" in inputs:
                inputs["text"] = negative_prompt
            elif title == _WIDTH_NODE_TITLE:
                if "width" in inputs:
                    inputs["width"] = w
                if "height" in inputs:
                    inputs["height"] = h
            elif title == _LORA_NODE_TITLE and self.config.loras:
                lora = self.config.loras[0]  # first LoRA
                if "lora_name" in inputs:
                    inputs["lora_name"] = lora.path
                if "strength_model" in inputs:
                    inputs["strength_model"] = lora.strength
            elif title == _REF_IMAGE_NODE_TITLE and reference_image_b64:
                if "image" in inputs:
                    inputs["image"] = reference_image_b64

        return workflow

    def _submit_and_wait(self, workflow: dict) -> bytes:
        """Submit a workflow to ComfyUI and poll until the image is ready."""
        base = self._model_cfg.base_url.rstrip("/")
        client_id = str(uuid.uuid4())

        # Queue the prompt
        resp = httpx.post(
            f"{base}/prompt",
            json={"prompt": workflow, "client_id": client_id},
            timeout=30,
        )
        resp.raise_for_status()
        prompt_id = resp.json()["prompt_id"]

        # Poll for completion
        deadline = time.time() + self._model_cfg.timeout
        while time.time() < deadline:
            time.sleep(self._model_cfg.poll_interval)
            history = httpx.get(f"{base}/history/{prompt_id}", timeout=10).json()
            if prompt_id in history:
                outputs = history[prompt_id].get("outputs", {})
                # Find first image output
                for node_output in outputs.values():
                    images = node_output.get("images", [])
                    if images:
                        img_info = images[0]
                        img_resp = httpx.get(
                            f"{base}/view",
                            params={"filename": img_info["filename"],
                                    "subfolder": img_info.get("subfolder", ""),
                                    "type": img_info.get("type", "output")},
                            timeout=30,
                        )
                        img_resp.raise_for_status()
                        return img_resp.content

        raise TimeoutError(f"ComfyUI job {prompt_id} did not complete within {self._model_cfg.timeout}s")

    async def render_async(
        self,
        panel: Panel,
        output_dir: Path,
        output_cfg: OutputConfig,
        reference_images: dict[str, Path] | None = None,
        negative_prompt: str = "",
    ) -> RenderResult:
        import asyncio, base64
        return await asyncio.get_event_loop().run_in_executor(
            None, self._render_sync, panel, output_dir, output_cfg, reference_images, negative_prompt
        )

    def _render_sync(
        self,
        panel: Panel,
        output_dir: Path,
        output_cfg: OutputConfig,
        reference_images: dict[str, Path] | None,
        negative_prompt: str = "",
    ) -> RenderResult:
        import base64
        workflow = self._load_workflow()

        # Pick first character's reference image
        ref_b64: str | None = None
        if reference_images:
            for char_name in panel.characters_present:
                ref_path = reference_images.get(char_name)
                if ref_path and ref_path.exists():
                    ref_b64 = base64.b64encode(ref_path.read_bytes()).decode()
                    break

        workflow = self._patch_workflow(workflow, panel, output_cfg, negative_prompt, ref_b64)
        image_bytes = self._submit_and_wait(workflow)

        out_path = output_dir / f"panel_{panel.panel_number:04d}.{output_cfg.format}"
        out_path.write_bytes(image_bytes)

        return RenderResult(
            panel_number=panel.panel_number,
            image_path=out_path,
            backend_used=self.model_spec.name,
            prompt_used=panel.visual_prompt,
            metadata={"ref_image_used": ref_b64 is not None, "loras": len(self.config.loras)},
        )

In [ ]:
# Construction and workflow patching tests (no server needed)
import json
from manhualizer.render import MODELS
from manhualizer.config import RendererConfig, LoRAConfig, OutputConfig
from manhualizer.models import Panel
from manhualizer.renderers.comfyui import ComfyUIRenderer

renderer = ComfyUIRenderer(MODELS["comfyui"], RendererConfig())
assert renderer.model_spec.capabilities.lora
assert renderer.model_spec.capabilities.reference_images
assert renderer.model_spec.capabilities.multi_image_input

# Patch a minimal mock workflow
mock_workflow = {
    "1": {"_meta": {"title": "positive_prompt"}, "inputs": {"text": "old prompt"}},
    "2": {"_meta": {"title": "negative_prompt"}, "inputs": {"text": ""}},
    "3": {"_meta": {"title": "empty_latent_image"}, "inputs": {"width": 512, "height": 512}},
}
renderer._workflow_template = mock_workflow

panel = Panel(panel_number=1, scene_id="s1", location="Forest",
              action_description="walks", visual_prompt="manhua style, forest scene")
out_cfg = OutputConfig(width=896, height=1152)

patched = renderer._patch_workflow(
    json.loads(json.dumps(mock_workflow)), panel, out_cfg, negative_prompt="blurry"
)
assert patched["1"]["inputs"]["text"] == "manhua style, forest scene"
assert patched["2"]["inputs"]["text"] == "blurry"
assert patched["3"]["inputs"]["width"] == 896
assert patched["3"]["inputs"]["height"] == 1152

print("ComfyUIRenderer OK")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()